# quantkit — Day 1 runner

Thin runner only: all logic lives in the `quantkit` package. Runtime → Change runtime type → **L4**.

In [ ]:
!nvidia-smi --query-gpu=name,driver_version,clocks.max.sm,power.limit --format=csv
import torch; assert 'L4' in torch.cuda.get_device_name(0), 'Not an L4 — switch runtime before collecting numbers'

In [ ]:
# Persist HF cache, checkpoints and results on Drive so a disconnect costs nothing.
from google.colab import drive
drive.mount('/content/drive')
import os
WORK = '/content/drive/MyDrive/quantkit'
os.makedirs(WORK, exist_ok=True)
os.environ['HF_HOME'] = f'{WORK}/hf_cache'

In [ ]:
# Llama-3.2 is gated: accept the license on its HF page first, and store your token
# in Colab Secrets (key icon) as HF_TOKEN.
from google.colab import userdata
from huggingface_hub import login
login(userdata.get('HF_TOKEN'))

In [ ]:
# Replace with your repo URL once you push it.
REPO = 'https://github.com/<your-username>/quantkit.git'
!rm -rf /content/quantkit && git clone -q $REPO /content/quantkit
%cd /content/quantkit
!pip install -q -e '.[dev]'

In [ ]:
!python -m pytest -q

## Quick sanity pass (a few windows, ~1 min)
Confirms the pipeline before spending time on full evals.

In [ ]:
!python scripts/quantize_eval.py --method fp16 --datasets wikitext2 --max-windows 8 --results-dir $WORK/results/sanity
!python scripts/quantize_eval.py --method rtn --group-size 128 --datasets wikitext2 --max-windows 8 --results-dir $WORK/results/sanity

## Day 1 full runs
fp16 baseline, RTN g128, RTN per-channel, and a 3-bit fake-quant ablation.

In [ ]:
R = f'{WORK}/results/day1'
!python scripts/quantize_eval.py --method fp16 --results-dir $R
!python scripts/quantize_eval.py --method rtn --group-size 128 --save $WORK/ckpt/rtn-g128 --results-dir $R
!python scripts/quantize_eval.py --method rtn --group-size -1 --results-dir $R
!python scripts/quantize_eval.py --method rtn --bits 3 --group-size 128 --fake --results-dir $R

In [ ]:
# Reload the saved int4 checkpoint: ppl must match the rtn-w4-g128 run exactly.
!python scripts/quantize_eval.py --load $WORK/ckpt/rtn-g128 --datasets wikitext2 --results-dir $R

In [ ]:
import json, glob
rows = [json.load(open(f)) for f in sorted(glob.glob(f'{R}/*.json'))]
print(f"{'run':28s} {'wikitext2':>10s} {'c4':>10s}")
for r in rows:
    p = r['perplexity']
    print(f"{r['label']:28s} {p.get('wikitext2',{}).get('ppl',float('nan')):10.3f} {p.get('c4',{}).get('ppl',float('nan')):10.3f}")

## Check now whether Nsight Compute works here (needed day 4–5)
If this prints `ERR_NVGPUCTRPERM`, profiling counters are blocked on Colab; plan on timing + bandwidth math instead.

In [ ]:
%%writefile /tmp/ncu_probe.py
import torch
a = torch.randn(1024, 1024, device='cuda', dtype=torch.float16)
(a @ a).sum().item()

In [ ]:
!which ncu || echo 'ncu not on PATH (try /usr/local/cuda/bin/ncu)'
!(ncu --version && ncu --launch-count 1 --section SpeedOfLight python /tmp/ncu_probe.py) 2>&1 | tail -20